# RAG Mini Assignment — Module 1

### Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers

### Write a short knowledge-source document about RAG

In [ ]:
rag_overview = """Retrieval-Augmented Generation (RAG) is an AI architecture that combines information retrieval with large language models to produce answers that are grounded in external knowledge. Traditional large language models are limited because they only know information available up to their training cutoff, cannot access private company documents, and often hallucinate when they lack reliable information. Hallucination happens when a model generates an answer that is incorrect, fabricated, unsupported, or outdated, usually because the required facts were missing from training data, the question was ambiguous, or the model simply predicts plausible text rather than verified facts.

RAG solves these problems by retrieving relevant documents before generating a response. A complete RAG pipeline includes a document loader, a chunking step that splits documents into smaller pieces, an embedding model that converts text into vectors, a vector database that stores those vectors, a retriever that finds the most relevant chunks for a given question, and a language model that generates the final answer using the retrieved context. Compared to fine-tuning, RAG is cheaper, faster to update, and relies on dynamic external knowledge rather than retraining the model itself.

RAG is used in many enterprise applications such as HR chatbots, customer support systems, banking assistants, healthcare tools, legal research, and internal knowledge search engines. Although RAG greatly reduces hallucinations and allows access to private and up to date information, it depends heavily on retrieval quality, requires additional infrastructure such as a vector database, and can add extra latency compared to a standalone language model."""

with open("rag_notes.txt", "w") as f:
    f.write(rag_overview)

### Read the file back and split it into retrieval-sized chunks

In [1]:
with open("rag_notes.txt", "r") as f:
    loaded_text = f.read()

rag_chunks = [piece.strip().replace("\n", " ") for piece in loaded_text.split(". ") if piece.strip()]
rag_chunks

['Retrieval-Augmented Generation (RAG) is an AI architecture that combines information retrieval with large language models to produce answers that are grounded in external knowledge',
 'Traditional large language models are limited because they only know information available up to their training cutoff, cannot access private company documents, and often hallucinate when they lack reliable information',
 'Hallucination happens when a model generates an answer that is incorrect, fabricated, unsupported, or outdated, usually because the required facts were missing from training data, the question was ambiguous, or the model simply predicts plausible text rather than verified facts.  RAG solves these problems by retrieving relevant documents before generating a response',
 'A complete RAG pipeline includes a document loader, a chunking step that splits documents into smaller pieces, an embedding model that converts text into vectors, a vector database that stores those vectors, a retriev

### Embed each chunk with a Sentence Transformer

In [1]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embed_model.encode(rag_chunks)
chunk_vectors.shape

(6, 384)


### Index the chunk embeddings with FAISS

In [1]:
import faiss
import numpy as np

vec_dim = chunk_vectors.shape[1]
faiss_index = faiss.IndexFlatL2(vec_dim)
faiss_index.add(np.array(chunk_vectors))
faiss_index.ntotal

6


### Check the installed `transformers` version, then load a generator model

In [1]:
import transformers
print(transformers.__version__)

5.13.1


In [1]:
from transformers import pipeline

answer_generator = pipeline("text-generation", model="google/flan-t5-base")

Note: T5ForConditionalGeneration is an encoder-decoder model and isn't in the standard causal-LM text-generation list, but the pipeline wrapper still runs it fine for this exercise.


### Retrieval helper: find the most relevant chunks for a question

In [ ]:
def retrieve_context(question, top_k=2):
    q_vec = embed_model.encode([question])
    _, top_idx = faiss_index.search(np.array(q_vec), top_k)
    return [rag_chunks[i] for i in top_idx[0]]

### Generation helper: answer a question using the retrieved context

In [ ]:
def generate_answer(question, context_text):
    gen_prompt = f"Answer the question using the context.\n\nContext: {context_text}\n\nQuestion: {question}"
    result = answer_generator(gen_prompt, max_new_tokens=100)
    return result[0]["generated_text"]

### Ask five questions and show the retrieved context alongside each generated answer

In [1]:
sample_questions = [
    "What is Retrieval-Augmented Generation?",
    "Why do large language models hallucinate?",
    "What are the components of a complete RAG pipeline?",
    "How does RAG compare to fine-tuning?",
    "What are the limitations of RAG?",
]

for q in sample_questions:
    retrieved = retrieve_context(q)
    context_text = " ".join(retrieved)
    answer = generate_answer(q, context_text)

    print("Question:", q)
    print("Retrieved Context:", context_text)
    print("Generated Answer:", answer)
    print("-" * 80)

Question: What is Retrieval-Augmented Generation?
Retrieved Context: Retrieval-Augmented Generation (RAG) is an AI architecture that combines information retrieval with large language models to produce answers that are grounded in external knowledge Hallucination happens when a model generates an answer that is incorrect, fabricated, unsupported, or outdated...
Generated Answer: Retrieval-Augmented Generation (RAG) is an AI architecture that combines information retrieval with large language models to produce answers that are grounded in external knowledge.
--------------------------------------------------------------------------------
Question: Why do large language models hallucinate?
Retrieved Context: Traditional large language models are limited because they only know information available up to their training cutoff, cannot access private company documents, and often hallucinate when they lack reliable information...
Generated Answer: They lack reliable information and the requi